<a href="https://colab.research.google.com/github/marcelofdariva/GUIRAO/blob/Guirao-codigo/GUIRAO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xlsxwriter
import pandas as pd
import numpy as np
from datetime import datetime

# --- CONFIGURACIÓN INICIAL AUTOMATIZADA ---
# Forzamos la zona horaria local para que el servidor (en UTC) no salte de día a la noche.
fecha_hoy = pd.Timestamp.now(tz='America/Argentina/Mendoza').normalize().tz_localize(None)
fecha_limite = pd.to_datetime('2026-09-30')

# Arma el nombre del archivo dinámicamente.
fecha_str = fecha_hoy.strftime('%d-%m-%Y')# @title Texto de título predeterminado

# ATENCIÓN: Si tu archivo empieza con mayúscula, cambiá la "s" por "S" en la línea de abajo:
archivo_entrada = f'stock al {fecha_str}.csv'
archivo_salida = f'stock al {fecha_str}.xlsx'

print(f"Buscando archivo '{archivo_entrada}' e iniciando procesamiento...")

# 1 y 2. IMPORTACIÓN Y TIPOS DE DATOS INICIALES
# Forzamos desde la carga que estas columnas sean estrictamente texto (str)
try:
    df = pd.read_csv(
        archivo_entrada,
        sep=',',
        encoding='utf-8',
        dtype={'Codigo': str, 'cost': str, 'Cantidad': str}
    )
except FileNotFoundError:
    print(f"\n❌ ERROR: No se encontró el archivo '{archivo_entrada}'. Verificá si empieza con mayúscula o minúscula.")
    raise

total_registros_iniciales = len(df)

# 3 y 4. CONVERSIÓN DE COST / CANTIDAD
if df['cost'].dtype == object:
    df['cost'] = df['cost'].str.replace(',', '.').astype(float)
else:
    df['cost'] = df['cost'].astype(float)

df['Cantidad'] = pd.to_numeric(df['Cantidad'], errors='coerce').fillna(0)

# 5. VALOR INVENTARIO
idx_cost = df.columns.get_loc('cost')
df.insert(idx_cost + 1, 'Valor inventario', df['cost'] * df['Cantidad'])

# 6. EXTRACCIÓN DE ATRIBUTOS
atributos_split = df['atributos'].str.split('_', expand=True)
df['Propiedad'] = atributos_split[0] if 0 in atributos_split.columns else None
df['Lote'] = atributos_split[1] if 1 in atributos_split.columns else None
vencimiento_str = atributos_split[2] if 2 in atributos_split.columns else None

# 7. CONVERSIÓN DE FECHAS
df['Vencimiento'] = pd.to_datetime(vencimiento_str, dayfirst=True, errors='coerce')
fechas_validas = df['Vencimiento'].notna().sum()
fechas_invalidas = df['Vencimiento'].isna().sum()

# 8. ORDENAMIENTO DE DATOS MAESTROS
df.sort_values(by=['Producto', 'Vencimiento'], ascending=[True, True], inplace=True)

# --- CREACIÓN DE HOJAS DE ANÁLISIS ---

# 11. HOJA "TD Categorias"
td_categorias = df.groupby('Categoria').agg(
    Suma_Valor_Inventario=('Valor inventario', 'sum'),
    Recuento_Codigo=('Codigo', 'count')
).reset_index()
td_categorias.sort_values('Suma_Valor_Inventario', ascending=False, inplace=True)

# Fila TOTAL para TD Categorias
total_td = pd.DataFrame({
    'Categoria': ['TOTAL'],
    'Suma_Valor_Inventario': [td_categorias['Suma_Valor_Inventario'].sum()],
    'Recuento_Codigo': [td_categorias['Recuento_Codigo'].sum()]
})
td_categorias = pd.concat([td_categorias, total_td], ignore_index=True)

# 12. HOJA "Prox Vencer"
columnas_filtro = ['Vencimiento', 'Codigo', 'Producto', 'Posicion', 'Valor inventario', 'Cantidad']
prox_vencer = df[(df['Vencimiento'] >= fecha_hoy) & (df['Vencimiento'] <= fecha_limite)][columnas_filtro].copy()
prox_vencer.sort_values(by=['Vencimiento', 'Producto'], ascending=[True, True], inplace=True)
total_prox_vencer = len(prox_vencer)

if total_prox_vencer > 0:
    fila_total_prox = pd.DataFrame({
        'Vencimiento': ['TOTAL'], 'Codigo': [''], 'Producto': [''], 'Posicion': [''],
        'Valor inventario': [prox_vencer['Valor inventario'].sum()],
        'Cantidad': [prox_vencer['Cantidad'].sum()]
    })
    prox_vencer = pd.concat([prox_vencer, fila_total_prox], ignore_index=True)

# 13. HOJA "Vencidos"
vencidos = df[df['Vencimiento'] < fecha_hoy][columnas_filtro].copy()
vencidos.sort_values(by=['Vencimiento', 'Producto'], ascending=[True, True], inplace=True)
total_vencidos = len(vencidos)

if total_vencidos > 0:
    fila_total_venc = pd.DataFrame({
        'Vencimiento': ['TOTAL'], 'Codigo': [''], 'Producto': [''], 'Posicion': [''],
        'Valor inventario': [vencidos['Valor inventario'].sum()],
        'Cantidad': [vencidos['Cantidad'].sum()]
    })
    vencidos = pd.concat([vencidos, fila_total_venc], ignore_index=True)

# --- 9 y 10. EXPORTACIÓN Y FORMATOS GLOBALES ---
print("Aplicando reglas de formato y escribiendo Excel...")
writer = pd.ExcelWriter(archivo_salida, engine='xlsxwriter', datetime_format='dd/mm/yyyy')
workbook = writer.book

# Definición de formatos
formato_moneda = workbook.add_format({'num_format': '$#,##0.00'})
formato_numero = workbook.add_format({'num_format': '#,##0'})
formato_fecha = workbook.add_format({'num_format': 'dd/mm/yyyy'})
formato_texto = workbook.add_format({'num_format': '@'})
formato_encabezado = workbook.add_format({'bold': True, 'bottom': 1, 'bg_color': '#F2F2F2'})

def aplicar_formato(df_hoja, nombre_hoja, ocultar_datos=False):
    df_hoja.to_excel(writer, sheet_name=nombre_hoja, index=False)
    worksheet = writer.sheets[nombre_hoja]

    # Encontrar última fila de datos (para no filtrar el TOTAL)
    ultima_fila = len(df_hoja) - 1 if not df_hoja.empty and df_hoja.iloc[-1].iloc[0] == 'TOTAL' else len(df_hoja)

    # Congelar primera fila y filtro automático
    worksheet.freeze_panes(1, 0)
    if ultima_fila > 0:
        worksheet.autofilter(0, 0, ultima_fila, len(df_hoja.columns)-1)

    # Formateo de columnas
    for col_num, col_name in enumerate(df_hoja.columns):
        worksheet.write(0, col_num, col_name, formato_encabezado)

        # Autoajuste dinámico de ancho
        max_len = max(df_hoja[col_name].astype(str).map(len).max(), len(col_name)) + 2

        if col_name in ['cost', 'Valor inventario', 'Suma_Valor_Inventario']:
            worksheet.set_column(col_num, col_num, max_len, formato_moneda)
        elif col_name in ['Cantidad', 'Recuento_Codigo']:
            worksheet.set_column(col_num, col_num, max_len, formato_numero)
        elif col_name == 'Vencimiento':
            worksheet.set_column(col_num, col_num, max_len, formato_fecha)
        elif col_name == 'Codigo':
            worksheet.set_column(col_num, col_num, max_len, formato_texto)
        else:
            worksheet.set_column(col_num, col_num, max_len)

        # 9. VISIBILIDAD: Ocultar columnas en hoja Datos
        columnas_visibles_datos = ['Codigo', 'Producto', 'Vencimiento', 'Posicion', 'Cantidad']
        if ocultar_datos and col_name not in columnas_visibles_datos:
            worksheet.set_column(col_num, col_num, options={'hidden': True})

# Aplicación a todas las pestañas solicitadas
aplicar_formato(df, 'Datos', ocultar_datos=True)
aplicar_formato(td_categorias, 'TD Categorias')
if total_prox_vencer > 0:
    aplicar_formato(prox_vencer, 'Prox Vencer')
if total_vencidos > 0:
    aplicar_formato(vencidos, 'Vencidos')

writer.close()

# --- 14 y 15. VALIDACIONES Y CONTROL DE CALIDAD ---
print("\n" + "="*40)
print(f"📊 REPORTE DE CONTROL DE CALIDAD AL {fecha_hoy.strftime('%d/%m/%Y')}")
print("="*40)
print(f"✓ Cantidad total de registros procesados: {total_registros_iniciales}")
print(f"✓ Cantidad de registros en Prox Vencer: {total_prox_vencer}")
print(f"✓ Cantidad de registros en Vencidos: {total_vencidos}")
print(f"✓ Fechas convertidas correctamente: {fechas_validas}")
print(f"✓ Fechas descartadas por formato inválido o NULL: {fechas_invalidas}")
print("-" * 40)
print("✓ Archivo Excel '.xlsx' compilado exitosamente. Listo para utilizar.")

Buscando archivo 'stock al 28-07-2026.csv' e iniciando procesamiento...
Aplicando reglas de formato y escribiendo Excel...

📊 REPORTE DE CONTROL DE CALIDAD AL 28/07/2026
✓ Cantidad total de registros procesados: 2176
✓ Cantidad de registros en Prox Vencer: 22
✓ Cantidad de registros en Vencidos: 70
✓ Fechas convertidas correctamente: 1435
✓ Fechas descartadas por formato inválido o NULL: 741
----------------------------------------
✓ Archivo Excel '.xlsx' compilado exitosamente. Listo para utilizar.
